# Day 5 — Distributed thinking, neural networks and a toy GAN

All data are synthetic. Run from top to bottom. Work in pairs and pause after each result to explain its meaning. Use TEACHING_GUIDE.md and TASK_CARDS.md for timing. Optional sections are marked. Numerical outputs are examples, not evidence about real operations.

## 1. Map, combine and reduce on a laptop

A local simulation divides records into partitions, computes partial totals, then combines them. This illustrates the idea of distributed aggregation; it is not Hadoop or Spark execution. Hadoop includes HDFS storage, YARN resource management and MapReduce processing. Spark uses a different execution engine and can work with Hadoop storage/resource infrastructure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
rows=[('A',10),('B',20),('A',5),('C',7),('B',3),('C',8)]
partitions=[rows[:3],rows[3:]]
partials=[]
for partition in partitions:
    subtotal=defaultdict(int)
    for key,value in partition: subtotal[key]+=value
    partials.append(dict(subtotal))
combined=defaultdict(int)
for partial in partials:
    for key,value in partial.items(): combined[key]+=value
print('Partial totals:',partials)
print('Combined totals:',dict(sorted(combined.items())))
assert dict(combined)=={'A':15,'B':23,'C':15}

## 2. A small neural network

Two hidden layers form a small multilayer perceptron. Scaling is fitted on training data only. This local synthetic example is not big-data training and uses no GPU. Compare with a linear decision boundary. Architectures are fixed before test evaluation; do not choose a new architecture from test results.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
X,y=make_moons(n_samples=600,noise=.22,random_state=55)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
linear=make_pipeline(StandardScaler(),LogisticRegression()).fit(X_train,y_train)
network=make_pipeline(StandardScaler(),MLPClassifier(hidden_layer_sizes=(16,8),max_iter=1200,early_stopping=True,n_iter_no_change=30,random_state=42)).fit(X_train,y_train)
print('Linear test accuracy:',round(accuracy_score(y_test,linear.predict(X_test)),3))
print('Neural-network test accuracy:',round(accuracy_score(y_test,network.predict(X_test)),3))
print('Network training iterations:',network[-1].n_iter_)
plt.figure(figsize=(6,3));plt.plot(network[-1].loss_curve_);plt.xlabel('Epoch');plt.ylabel('Training loss');plt.title('Loss is not a test-set metric');plt.tight_layout();plt.show()

## 3. A tiny adversarial generator — optional demonstration

This is an actual one-dimensional adversarial training loop, not an image GAN or a deep network. Generator G(z)=mu+sigma*z produces a Gaussian. Discriminator uses logistic scores of x and x squared. Alternate discriminator and generator updates. The exercise illustrates competing objectives and instability, not useful production synthesis.

In [ ]:
from scipy.special import expit
rng=np.random.default_rng(55)
mu,log_sigma=-1.0,0.0
w=np.zeros(3)
log=[]
for step in range(2500):
    real=rng.normal(2,.6,256)
    z=rng.normal(size=256)
    fake=mu+np.exp(log_sigma)*z
    phi_r=np.column_stack([np.ones(len(real)),real,real**2])
    phi_f=np.column_stack([np.ones(len(fake)),fake,fake**2])
    pr,pf=expit(phi_r@w),expit(phi_f@w)
    w+=.03*((1-pr)@phi_r/len(real)-pf@phi_f/len(fake))
    z=rng.normal(size=256);sigma=np.exp(log_sigma);fake=mu+sigma*z
    score=w[0]+w[1]*fake+w[2]*fake**2
    gradient=(expit(score)-1)*(w[1]+2*w[2]*fake)
    mu-=.01*gradient.mean()
    log_sigma-=.01*np.mean(gradient*sigma*z)
    log_sigma=float(np.clip(log_sigma,-2,1))
    if step%250==0:log.append((step,mu,np.exp(log_sigma)))
print(pd.DataFrame(log,columns=['step','generator_mean','generator_sd']).round(3).to_string(index=False))
print('Target mean/sd:',2,.6,'Final generator:',round(mu,3),round(np.exp(log_sigma),3))
generated=mu+np.exp(log_sigma)*rng.normal(size=2000)
reference=rng.normal(2,.6,2000)
plt.figure(figsize=(7,3));plt.hist(reference,bins=30,alpha=.5,label='Reference');plt.hist(generated,bins=30,alpha=.5,label='Generator');plt.xlabel('Synthetic scalar value');plt.ylabel('Count');plt.legend();plt.tight_layout();plt.show()
print('An overlapping histogram is not a privacy or deployment-quality guarantee.')

# Exercise solutions
Run after the worked examples. These experiments use training/development data where model choices are involved. See ANSWER_KEY.md for interpretations.

In [ ]:
extra_rows=rows+[('A',4)]
answer=defaultdict(int)
for key,value in extra_rows:answer[key]+=value
print('Updated totals:',dict(sorted(answer.items())))
print('Region A mean from verified aggregate:',250000/5000)
print('Region B mean from verified aggregate:',255000/5000)
print('Generator spread ratio to target:',round(np.exp(log_sigma)/.6,3))
print('The generator substantially under-represents spread in this run.')
